# Análise de operações financeiras - Nível 1
## Parte A - Análise determinística com Pandas

In [1]:
import json
from pathlib import Path
import pandas as pd

In [ ]:
caminho_dados = Path("../dados/dados_nivel_1.json")
Path.cwd()
caminho_dados.exists()

True

In [20]:

with open(caminho_dados, "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]

df = pd.DataFrame(dados["operacoes"])
df


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [21]:
df.isna().sum()


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [22]:
df["id"].duplicated().sum()

np.int64(1)

In [23]:
df["moeda"].value_counts(dropna=False)
df["canal"].value_counts(dropna=False)
df["tipo"].value_counts(dropna=False)

tipo
transferencia_enviada     11
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

In [ ]:
df["cliente_id"].nunique()

# 20 operacoes, e 6 clientes

6

In [29]:
df[df["id"].duplicated(keep=False)].sort_values("id")

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [40]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["data"].dtype

dtype('<M8[us]')

In [47]:
df = df.drop_duplicates().copy()

df["data"] = pd.to_datetime(df["data"], errors="coerce")

df["data_ausente"] = df["data"].isna()

In [48]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False


In [49]:
print("Quantidade de registros:", len(df))
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Clientes únicos:", df["cliente_id"].nunique())

Quantidade de registros: 19
Duplicidades integrais: 0
Datas ausentes: 1
Clientes únicos: 6


In [50]:
colunas_categoricas = ["canal", "tipo"]

for coluna in colunas_categoricas:
    df[coluna] = df[coluna].str.strip().str.lower()

In [51]:
for coluna in colunas_categoricas:
    print(f"\nValores de {coluna}:")
    print(df[coluna].value_counts(dropna=False))


Valores de canal:
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

Valores de tipo:
tipo
transferencia_enviada     10
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64


In [52]:
df["moeda"] = df["moeda"].str.strip().str.upper()

In [53]:
df["moeda"].value_counts(dropna=False)

moeda
BRL    18
USD     1
Name: count, dtype: int64

In [54]:
df["valor"].describe()

count       19.000000
mean     11194.736842
std       8085.272873
min       1400.000000
25%       4050.000000
50%       8800.000000
75%      17250.000000
max      27000.000000
Name: valor, dtype: float64

In [55]:
df["valor_brl"] = df["valor"]

In [ ]:
df["valor_brl"] = df["valor"].astype(float)

# Identificar operações em USD
mascara_usd = df["moeda"] == "USD"

# Converter USD para BRL
df.loc[mascara_usd, "valor_brl"] = (
    df.loc[mascara_usd, "valor"].astype(float)
    * float(taxa_cambio_usd_brl)
)

In [61]:
df
# versão final do dataframe após as normalizações

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False,25900.0
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False,27000.0
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,17200.0
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,15200.0
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False,16100.0
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False,3800.0


In [62]:
print("Quantidade de registros:", len(df))
print("Clientes únicos:", df["cliente_id"].nunique())
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Valores BRL ausentes:", df["valor_brl"].isna().sum())
print("Moedas encontradas:", df["moeda"].unique())

Quantidade de registros: 19
Clientes únicos: 6
Duplicidades integrais: 0
Datas ausentes: 1
Valores BRL ausentes: 0
Moedas encontradas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: str


In [63]:
volume_por_cliente = (
    df.groupby("cliente_id", as_index=False)
      .agg(volume_total_brl=("valor_brl", "sum"))
      .sort_values("volume_total_brl", ascending=False)
)

In [64]:
volume_por_cliente


,cliente_id,volume_total_brl
3,CLI-A-4,79500.0
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [65]:
quantidade_por_canal = (
    df.groupby("canal", as_index=False)
      .agg(quantidade_operacoes=("id", "count"))
      .sort_values("quantidade_operacoes", ascending=False)
)

In [66]:
quantidade_por_canal

,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


In [67]:
total_base = df["valor_brl"].sum()
total_agregado = volume_por_cliente["volume_total_brl"].sum()

print("Total da base:", total_base)
print("Total agregado:", total_agregado)
print("Totais iguais:", total_base == total_agregado)

Total da base: 265500.0
Total agregado: 265500.0
Totais iguais: True


In [68]:
print(
    "Contagem por canal igual ao total de registros:",
    quantidade_por_canal["quantidade_operacoes"].sum() == len(df)
)

Contagem por canal igual ao total de registros: True


In [69]:
df_regra_1 = df[df["data"].notna()].copy()

In [70]:
resumo_regra_1 = (
    df_regra_1
    .groupby(["cliente_id", "data"], as_index=False)
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_operacoes_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
)

In [71]:
resumo_regra_1["sinalizado_regra_1"] = (
    (resumo_regra_1["quantidade_operacoes"] >= 3)
    & (resumo_regra_1["soma_operacoes_brl"] > 50_000)
    & (resumo_regra_1["maior_operacao_brl"] < 20_000)
)

In [72]:
casos_regra_1 = resumo_regra_1[
    resumo_regra_1["sinalizado_regra_1"]
].copy()

casos_regra_1

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


In [73]:
operacoes_sinalizadas_regra_1 = df_regra_1.merge(
    casos_regra_1[["cliente_id", "data"]],
    on=["cliente_id", "data"],
    how="inner"
)

operacoes_sinalizadas_regra_1[
    [
        "id",
        "cliente_id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte"
    ]
]

,id,cliente_id,data,valor,moeda,valor_brl,canal,tipo,contraparte
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,18100.0,pix,transferencia_enviada,Alfa Comercio LTDA
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,17300.0,pix,transferencia_enviada,Alfa Comercio LTDA
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,18800.0,ted,transferencia_enviada,Beta Servicos ME


In [74]:
validacao_positiva = resumo_regra_1[
    (resumo_regra_1["cliente_id"] == "CLI-A-1")
    & (resumo_regra_1["data"] == pd.Timestamp("2026-03-09"))
]

validacao_positiva

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


In [75]:
assert len(validacao_positiva) == 1
assert validacao_positiva.iloc[0]["quantidade_operacoes"] == 3
assert validacao_positiva.iloc[0]["soma_operacoes_brl"] == 54_200
assert validacao_positiva.iloc[0]["maior_operacao_brl"] == 18_800
assert bool(validacao_positiva.iloc[0]["sinalizado_regra_1"]) is True

print("Caso positivo validado corretamente.")

Caso positivo validado corretamente.


In [76]:
validacao_negativa = resumo_regra_1[
    (resumo_regra_1["cliente_id"] == "CLI-A-3")
    & (resumo_regra_1["data"] == pd.Timestamp("2026-03-05"))
]

validacao_negativa

,cliente_id,data,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


In [77]:
assert len(validacao_negativa) == 1
assert validacao_negativa.iloc[0]["quantidade_operacoes"] == 3
assert validacao_negativa.iloc[0]["soma_operacoes_brl"] == 48_500
assert validacao_negativa.iloc[0]["maior_operacao_brl"] == 17_200
assert bool(validacao_negativa.iloc[0]["sinalizado_regra_1"]) is False

print("Caso negativo validado corretamente.")

Caso negativo validado corretamente.


In [103]:
import math

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["valor_brl"],
    64_800.0,
    rel_tol=1e-9,
    abs_tol=0.01
)

In [104]:
estatisticas_clientes = (
    df.groupby("cliente_id", as_index=False)
      .agg(
          quantidade_operacoes=("id", "count"),
          mediana_cliente_brl=("valor_brl", "median")
      )
)

estatisticas_clientes

,cliente_id,quantidade_operacoes,mediana_cliente_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [105]:
analise_regra_2 = df.merge(
    estatisticas_clientes,
    on="cliente_id",
    how="left"
)

In [106]:
analise_regra_2[
    [
        "id",
        "cliente_id",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl"
    ]
]

,id,cliente_id,valor_brl,quantidade_operacoes,mediana_cliente_brl
0,OP-0001,CLI-A-1,18100.0,4,17700.0
1,OP-0002,CLI-A-1,17300.0,4,17700.0
2,OP-0003,CLI-A-1,18800.0,4,17700.0
3,OP-0004,CLI-A-1,3300.0,4,17700.0
4,OP-0005,CLI-A-2,25900.0,2,26450.0
5,OP-0006,CLI-A-2,27000.0,2,26450.0
6,OP-0007,CLI-A-3,17200.0,3,16100.0
7,OP-0008,CLI-A-3,15200.0,3,16100.0
8,OP-0009,CLI-A-3,16100.0,3,16100.0
9,OP-0010,CLI-A-4,3800.0,4,5450.0


In [107]:
analise_regra_2["limite_5x_mediana_brl"] = (
    analise_regra_2["mediana_cliente_brl"] * 5
)

In [108]:
analise_regra_2["sinalizado_regra_2"] = (
    (analise_regra_2["quantidade_operacoes"] >= 4)
    & (
        analise_regra_2["valor_brl"]
        > analise_regra_2["limite_5x_mediana_brl"]
    )
)

In [109]:
casos_regra_2 = analise_regra_2[
    analise_regra_2["sinalizado_regra_2"]
].copy()

casos_regra_2[
    [
        "id",
        "cliente_id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl",
        "limite_5x_mediana_brl",
        "sinalizado_regra_2"
    ]
]

,id,cliente_id,data,valor,moeda,valor_brl,quantidade_operacoes,mediana_cliente_brl,limite_5x_mediana_brl,sinalizado_regra_2
12,OP-0013,CLI-A-4,2026-03-24,12000,USD,64800.0,4,5450.0,27250.0,True


In [110]:
validacao_positiva_regra_2 = analise_regra_2[
    analise_regra_2["id"] == "OP-0013"
]

validacao_positiva_regra_2[
    [
        "id",
        "cliente_id",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_cliente_brl",
        "limite_5x_mediana_brl",
        "sinalizado_regra_2"
    ]
]

,id,cliente_id,valor_brl,quantidade_operacoes,mediana_cliente_brl,limite_5x_mediana_brl,sinalizado_regra_2
12,OP-0013,CLI-A-4,64800.0,4,5450.0,27250.0,True


In [112]:
import math

assert len(validacao_positiva_regra_2) == 1
assert validacao_positiva_regra_2.iloc[0]["quantidade_operacoes"] == 4

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["mediana_cliente_brl"],
    5_450.0,
    abs_tol=0.01
)

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["limite_5x_mediana_brl"],
    27_250.0,
    abs_tol=0.01
)

assert math.isclose(
    validacao_positiva_regra_2.iloc[0]["valor_brl"],
    64_800.0,
    abs_tol=0.01
)

assert bool(
    validacao_positiva_regra_2.iloc[0]["sinalizado_regra_2"]
) is True

print("Caso positivo da Regra 2 validado corretamente.")

Caso positivo da Regra 2 validado corretamente.


In [113]:
clientes_nao_elegiveis = analise_regra_2[
    analise_regra_2["quantidade_operacoes"] < 4
]

assert not clientes_nao_elegiveis["sinalizado_regra_2"].any()

print("Clientes com menos de quatro operações não foram sinalizados.")

Clientes com menos de quatro operações não foram sinalizados.


In [114]:
clientes_regra_1 = (
    casos_regra_1[["cliente_id"]]
    .drop_duplicates()
    .assign(regra_1=True)
)

In [115]:
clientes_regra_2 = (
    casos_regra_2[["cliente_id"]]
    .drop_duplicates()
    .assign(regra_2=True)
)

In [116]:
clientes_sinalizados = clientes_regra_1.merge(
    clientes_regra_2,
    on="cliente_id",
    how="outer"
)

clientes_sinalizados[["regra_1", "regra_2"]] = (
    clientes_sinalizados[["regra_1", "regra_2"]]
    .fillna(False)
    .astype(bool)
)

clientes_sinalizados

,cliente_id,regra_1,regra_2
0,CLI-A-1,True,False
1,CLI-A-4,False,True


In [117]:
cliente_escolhido = "CLI-A-1"

In [118]:
operacoes_cliente_escolhido = df[
    df["cliente_id"] == cliente_escolhido
].copy()

operacoes_cliente_escolhido

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,3300.0


In [119]:
resumo_cliente_escolhido = {
    "cliente_id": cliente_escolhido,
    "quantidade_total_operacoes": len(operacoes_cliente_escolhido),
    "volume_total_brl": float(
        operacoes_cliente_escolhido["valor_brl"].sum()
    ),
    "regra_1": True,
    "regra_2": False
}

resumo_cliente_escolhido

{'cliente_id': 'CLI-A-1',
 'quantidade_total_operacoes': 4,
 'volume_total_brl': 57500.0,
 'regra_1': True,
 'regra_2': False}

In [120]:
dados_cliente_llm = operacoes_cliente_escolhido[
    [
        "id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte",
        "observacao"
    ]
].copy()

In [121]:
dados_cliente_llm["data"] = (
    dados_cliente_llm["data"]
    .dt.strftime("%Y-%m-%d")
)

In [122]:
operacoes_para_prompt = dados_cliente_llm.to_dict(
    orient="records"
)

In [123]:
operacoes_para_prompt

[{'id': 'OP-0001',
  'data': '2026-03-09',
  'valor': 18100,
  'moeda': 'BRL',
  'valor_brl': 18100.0,
  'canal': 'pix',
  'tipo': 'transferencia_enviada',
  'contraparte': 'Alfa Comercio LTDA',
  'observacao': ''},
 {'id': 'OP-0002',
  'data': '2026-03-09',
  'valor': 17300,
  'moeda': 'BRL',
  'valor_brl': 17300.0,
  'canal': 'pix',
  'tipo': 'transferencia_enviada',
  'contraparte': 'Alfa Comercio LTDA',
  'observacao': ''},
 {'id': 'OP-0003',
  'data': '2026-03-09',
  'valor': 18800,
  'moeda': 'BRL',
  'valor_brl': 18800.0,
  'canal': 'ted',
  'tipo': 'transferencia_enviada',
  'contraparte': 'Beta Servicos ME',
  'observacao': ''},
 {'id': 'OP-0004',
  'data': '2026-03-21',
  'valor': 3300,
  'moeda': 'BRL',
  'valor_brl': 3300.0,
  'canal': 'boleto',
  'tipo': 'pagamento',
  'contraparte': 'Gama Distribuidora',
  'observacao': ''}]

In [124]:
import json

In [125]:
contexto_cliente_json = json.dumps(
    {
        "resumo_cliente": resumo_cliente_escolhido,
        "operacoes": operacoes_para_prompt
    },
    ensure_ascii=False,
    indent=2
)

In [126]:
print(contexto_cliente_json)

{
  "resumo_cliente": {
    "cliente_id": "CLI-A-1",
    "quantidade_total_operacoes": 4,
    "volume_total_brl": 57500.0,
    "regra_1": true,
    "regra_2": false
  },
  "operacoes": [
    {
      "id": "OP-0001",
      "data": "2026-03-09",
      "valor": 18100,
      "moeda": "BRL",
      "valor_brl": 18100.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA",
      "observacao": ""
    },
    {
      "id": "OP-0002",
      "data": "2026-03-09",
      "valor": 17300,
      "moeda": "BRL",
      "valor_brl": 17300.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA",
      "observacao": ""
    },
    {
      "id": "OP-0003",
      "data": "2026-03-09",
      "valor": 18800,
      "moeda": "BRL",
      "valor_brl": 18800.0,
      "canal": "ted",
      "tipo": "transferencia_enviada",
      "contraparte": "Beta Servicos ME",
      "observacao": ""
    },
    {
      "id": "OP-0004

In [127]:
prompt_versao_1 = f"""
Analise as operações financeiras do cliente abaixo e produza um parecer
sobre o possível risco apresentado.

O cliente já foi sinalizado por uma regra determinística. Não refaça os
cálculos e utilize somente as informações fornecidas.

Dados do cliente:
{contexto_cliente_json}

Responda exclusivamente em JSON válido, seguindo esta estrutura:
{{
  "nivel_risco": "baixo, medio ou alto",
  "tipologia_suspeita": "descrição breve da possível tipologia",
  "red_flags": [
    "primeiro sinal de alerta",
    "segundo sinal de alerta"
  ],
  "justificativa": "justificativa objetiva baseada nos dados fornecidos"
}}
"""

In [130]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai


In [131]:
caminho_env = (
    Path(".env")
    if Path(".env").exists()
    else Path("../.env")
)

load_dotenv(dotenv_path=caminho_env)

True

In [132]:
api_key = os.getenv("GEMINI_API_KEY")

In [133]:
if not api_key:
    raise ValueError(
        "Chave do Gemini não encontrada no arquivo .env."
    )

print("Chave carregada com segurança.")

Chave carregada com segurança.


In [ ]:
client = genai.Client(api_key=api_key)

mmodelo = "gemini-3.5-flash-lite"

In [150]:
inicio = time.perf_counter()

resposta_versao_1, tentativas_versao_1 = gerar_com_tentativas(
    client=client,
    modelo=modelo,
    prompt=prompt_versao_1
)

tempo_versao_1 = time.perf_counter() - inicio

In [152]:
cliente_escolhido = "CLI-A-1"

caso_selecionado = casos_regra_1[
    casos_regra_1["cliente_id"] == cliente_escolhido
].iloc[0]

In [153]:
dados_cliente_llm = df.loc[
    df["cliente_id"] == cliente_escolhido,
    [
        "id",
        "data",
        "valor",
        "moeda",
        "valor_brl",
        "canal",
        "tipo",
        "contraparte",
        "observacao"
    ]
].copy()

dados_cliente_llm["data"] = (
    dados_cliente_llm["data"]
    .dt.strftime("%Y-%m-%d")
)

In [154]:
contexto_cliente = {
    "cliente_id": cliente_escolhido,
    "regra_acionada": "Regra 1",
    "criterios_regra": {
        "mesmo_cliente_e_data": True,
        "quantidade_minima_operacoes": 3,
        "soma_superior_a_brl": 50_000,
        "nenhuma_operacao_individual_maior_ou_igual_a_brl": 20_000
    },
    "resultado_calculado_pandas": {
        "data_do_alerta": caso_selecionado["data"].strftime(
            "%Y-%m-%d"
        ),
        "quantidade_operacoes": int(
            caso_selecionado["quantidade_operacoes"]
        ),
        "soma_operacoes_brl": float(
            caso_selecionado["soma_operacoes_brl"]
        ),
        "maior_operacao_brl": float(
            caso_selecionado["maior_operacao_brl"]
        )
    },
    "operacoes_do_cliente": dados_cliente_llm.to_dict(
        orient="records"
    )
}

In [155]:
contexto_cliente_json = json.dumps(
    contexto_cliente,
    ensure_ascii=False,
    indent=2
)

In [156]:
prompt_versao_1 = f"""
Você atua como analista de prevenção a riscos financeiros.

Produza um parecer preliminar sobre as operações do cliente apresentado
abaixo. O cliente já foi sinalizado por uma regra determinística
implementada em Pandas.

Use exclusivamente os dados fornecidos. Não invente informações, não
atribua culpa ao cliente e não afirme que ocorreu um crime. O parecer deve
apenas descrever os sinais de alerta observados e a necessidade, ou não,
de análise adicional.

Não refaça os cálculos. Considere os valores apresentados em
"resultado_calculado_pandas" como resultados já validados.

Contexto do caso:
{contexto_cliente_json}

Responda exclusivamente com um objeto JSON válido, sem Markdown, sem
blocos de código e sem qualquer texto antes ou depois do JSON.

Utilize exatamente esta estrutura:

{{
  "nivel_risco": "baixo, médio ou alto",
  "tipologia_suspeita": "descrição breve da possível tipologia ou comportamento observado",
  "red_flags": [
    "sinal de alerta diretamente sustentado pelos dados"
  ],
  "justificativa": "parecer objetivo fundamentado exclusivamente nos dados fornecidos"
}}

Regras para a resposta:

1. "nivel_risco" deve conter somente um destes valores:
   "baixo", "médio" ou "alto".
2. "tipologia_suspeita" deve ser uma string.
3. "red_flags" deve ser uma lista de strings.
4. "justificativa" deve explicar a classificação sem inventar dados.
5. A resposta deve indicar que o parecer é preliminar quando não houver
   informações suficientes para uma conclusão definitiva.
"""

In [157]:
modelo = "gemini-3.5-flash-lite"

inicio = time.perf_counter()

resposta_versao_1, tentativas_versao_1 = gerar_com_tentativas(
    client=client,
    modelo=modelo,
    prompt=prompt_versao_1
)

tempo_versao_1 = time.perf_counter() - inicio

In [158]:
print(resposta_versao_1.text)
print(f"Modelo: {modelo}")
print(f"Tentativas: {tentativas_versao_1}")
print(f"Tempo: {tempo_versao_1:.2f} segundos")

{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Fracionamento de valores em um mesmo dia para atingir montante superior a BRL 50.000,00 sem que nenhuma operação individual atinja BRL 20.000,00",
  "red_flags": [
    "Realização de 3 operações na mesma data (2026-03-09)",
    "Soma das operações na data atingiu BRL 54.200,00, superando o limite de BRL 50.000,00",
    "Nenhuma das operações individuais na data atingiu ou ultrapassou o patamar de BRL 20.000,00 (maior operação de BRL 18.800,00)"
  ],
  "justificativa": "Trata-se de um parecer preliminar, visto que não há informações suficientes para uma conclusão definitiva. O cliente CLI-A-1 foi sinalizado pela Regra 1 em decorrência de um padrão operacional na data 2026-03-09, onde foram registradas 3 transferências enviadas com soma total de BRL 54.200,00, sendo que individualmente nenhuma operação atingiu BRL 20.000,00. Há necessidade de análise adicional para verificar a motivação comercial das transações."
}
Modelo: gemini-3.5-f

In [159]:
niveis_risco_validos = {
    "baixo",
    "médio",
    "alto"
}

In [161]:
texto_resposta_v1 = resposta_versao_1.text.strip()

texto_resposta_v1 = (
    texto_resposta_v1
    .removeprefix("```json")
    .removeprefix("```")
    .removesuffix("```")
    .strip()
)

try:
    parecer_versao_1 = json.loads(texto_resposta_v1)
    resposta_json_valida_v1 = True
    print("Resposta convertida para JSON com sucesso.")

except json.JSONDecodeError as erro:
    parecer_versao_1 = None
    resposta_json_valida_v1 = False
    print(f"Resposta JSON malformada: {erro}")

Resposta convertida para JSON com sucesso.


In [164]:
campos_obrigatorios = {
    "nivel_risco",
    "tipologia_suspeita",
    "red_flags",
    "justificativa"
}

campos_ausentes = (
    campos_obrigatorios - set(parecer_versao_1.keys())
)

assert not campos_ausentes, (
    f"Campos obrigatórios ausentes: {campos_ausentes}"
)

print("Todos os campos obrigatórios estão presentes.")

Todos os campos obrigatórios estão presentes.


In [165]:
niveis_risco_validos = {
    "baixo",
    "médio",
    "alto"
}

assert parecer_versao_1["nivel_risco"] in niveis_risco_validos

assert isinstance(
    parecer_versao_1["tipologia_suspeita"],
    str
)

assert isinstance(
    parecer_versao_1["red_flags"],
    list
)

assert all(
    isinstance(item, str)
    for item in parecer_versao_1["red_flags"]
)

assert isinstance(
    parecer_versao_1["justificativa"],
    str
)

print("Estrutura da Versão 1 validada corretamente.")

Estrutura da Versão 1 validada corretamente.


In [169]:
CAMPOS_OBRIGATORIOS = {
    "nivel_risco",
    "tipologia_suspeita",
    "red_flags",
    "justificativa"
}

NIVEIS_VALIDOS = {"baixo", "médio", "alto"}


def converter_e_validar_resposta(resposta):
    texto_original = resposta.text.strip()

    texto_limpo = (
        texto_original
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )

    try:
        parecer = json.loads(texto_limpo)
    except json.JSONDecodeError as erro:
        return {
            "valido": False,
            "parecer": None,
            "texto_original": texto_original,
            "erro": f"JSON malformado: {erro}"
        }

    campos_ausentes = CAMPOS_OBRIGATORIOS - set(parecer.keys())

    if campos_ausentes:
        return {
            "valido": False,
            "parecer": parecer,
            "texto_original": texto_original,
            "erro": f"Campos ausentes: {campos_ausentes}"
        }

    if parecer["nivel_risco"] not in NIVEIS_VALIDOS:
        return {
            "valido": False,
            "parecer": parecer,
            "texto_original": texto_original,
            "erro": "Nível de risco inválido."
        }

    if not isinstance(parecer["tipologia_suspeita"], str):
        return {
            "valido": False,
            "parecer": parecer,
            "texto_original": texto_original,
            "erro": "tipologia_suspeita deve ser string."
        }

    if not isinstance(parecer["red_flags"], list) or not all(
        isinstance(item, str) for item in parecer["red_flags"]
    ):
        return {
            "valido": False,
            "parecer": parecer,
            "texto_original": texto_original,
            "erro": "red_flags deve ser uma lista de strings."
        }

    if not isinstance(parecer["justificativa"], str):
        return {
            "valido": False,
            "parecer": parecer,
            "texto_original": texto_original,
            "erro": "justificativa deve ser string."
        }

    return {
        "valido": True,
        "parecer": parecer,
        "texto_original": texto_original,
        "erro": None
    }


def extrair_metricas(resposta, tempo, tentativas):
    uso = getattr(resposta, "usage_metadata", None)

    return {
        "tempo_segundos": round(tempo, 2),
        "tentativas": tentativas,
        "tokens_entrada": getattr(
            uso, "prompt_token_count", None
        ),
        "tokens_saida": getattr(
            uso, "candidates_token_count", None
        ),
        "tokens_total": getattr(
            uso, "total_token_count", None
        )
    }


In [170]:
prompt_versao_2 = f"""
Você atua como analista de prevenção a riscos financeiros e deve elaborar
um parecer preliminar sobre um cliente previamente sinalizado por uma
regra determinística implementada em Pandas.

Limites da análise:

- Utilize exclusivamente os dados fornecidos.
- Não refaça os cálculos realizados pelo Pandas.
- Não invente contexto, histórico, intenção ou informações externas.
- Não afirme que ocorreu fraude, crime ou irregularidade comprovada.
- Diferencie um padrão observado de uma conclusão definitiva.
- Trate a tipologia apenas como uma hipótese compatível com os dados.
- Não recomende bloqueio, punição ou encerramento da conta.
- A justificativa deve mencionar os valores objetivos que geraram o alerta.

Contexto validado pelo Pandas:
{contexto_cliente_json}

Responda exclusivamente com um único objeto JSON válido, sem Markdown,
sem bloco de código e sem texto adicional.

A resposta deve seguir exatamente este formato:

{{
  "nivel_risco": "baixo | médio | alto",
  "tipologia_suspeita": "padrão compatível com possível tipologia, sem afirmar intenção",
  "red_flags": [
    "evidência objetiva 1",
    "evidência objetiva 2"
  ],
  "justificativa": "parecer preliminar fundamentado nos dados e com indicação das limitações"
}}

Requisitos obrigatórios:

1. Use somente "baixo", "médio" ou "alto" em "nivel_risco".
2. Apresente entre 2 e 4 red flags.
3. Cada red flag deve ser sustentada diretamente pelos dados.
4. Mencione na justificativa que a sinalização não comprova irregularidade.
5. Não atribua finalidade ou intenção às operações.
6. Não inclua campos além dos quatro solicitados.
"""

In [171]:
resultado_validacao_v1 = converter_e_validar_resposta(
    resposta_versao_1
)

parecer_versao_1 = resultado_validacao_v1["parecer"]

metricas_versao_1 = extrair_metricas(
    resposta=resposta_versao_1,
    tempo=tempo_versao_1,
    tentativas=tentativas_versao_1
)


In [172]:
inicio_v2 = time.perf_counter()

resposta_versao_2, tentativas_versao_2 = gerar_com_tentativas(
    client=client,
    modelo=modelo,
    prompt=prompt_versao_2
)

tempo_versao_2 = time.perf_counter() - inicio_v2

resultado_validacao_v2 = converter_e_validar_resposta(
    resposta_versao_2
)

parecer_versao_2 = resultado_validacao_v2["parecer"]

metricas_versao_2 = extrair_metricas(
    resposta=resposta_versao_2,
    tempo=tempo_versao_2,
    tentativas=tentativas_versao_2
)

In [173]:
comparacao_prompts = pd.DataFrame([
    {
        "versao": "Versão 1",
        "modelo": modelo,
        "json_valido": resultado_validacao_v1["valido"],
        "nivel_risco": (
            parecer_versao_1.get("nivel_risco")
            if parecer_versao_1 else None
        ),
        "quantidade_red_flags": (
            len(parecer_versao_1.get("red_flags", []))
            if parecer_versao_1 else 0
        ),
        **metricas_versao_1
    },
    {
        "versao": "Versão 2",
        "modelo": modelo,
        "json_valido": resultado_validacao_v2["valido"],
        "nivel_risco": (
            parecer_versao_2.get("nivel_risco")
            if parecer_versao_2 else None
        ),
        "quantidade_red_flags": (
            len(parecer_versao_2.get("red_flags", []))
            if parecer_versao_2 else 0
        ),
        **metricas_versao_2
    }
])


In [174]:
comparacao_prompts = pd.DataFrame([
    {
        "versao": "Versão 1",
        "modelo": modelo,
        "json_valido": resultado_validacao_v1["valido"],
        "nivel_risco": (
            parecer_versao_1.get("nivel_risco")
            if parecer_versao_1 else None
        ),
        "quantidade_red_flags": (
            len(parecer_versao_1.get("red_flags", []))
            if parecer_versao_1 else 0
        ),
        **metricas_versao_1
    },
    {
        "versao": "Versão 2",
        "modelo": modelo,
        "json_valido": resultado_validacao_v2["valido"],
        "nivel_risco": (
            parecer_versao_2.get("nivel_risco")
            if parecer_versao_2 else None
        ),
        "quantidade_red_flags": (
            len(parecer_versao_2.get("red_flags", []))
            if parecer_versao_2 else 0
        ),
        **metricas_versao_2
    }
])

In [175]:
raiz_projeto = Path.cwd()

if not (raiz_projeto / "ENTREGA.yaml").exists():
    if (raiz_projeto.parent / "ENTREGA.yaml").exists():
        raiz_projeto = raiz_projeto.parent

pasta_outputs = raiz_projeto / "outputs"
pasta_outputs.mkdir(exist_ok=True)

volume_por_cliente.to_csv(
    pasta_outputs / "nivel_1_volume_por_cliente.csv",
    index=False
)

quantidade_por_canal.to_csv(
    pasta_outputs / "nivel_1_quantidade_por_canal.csv",
    index=False
)

casos_regra_1.to_csv(
    pasta_outputs / "nivel_1_casos_regra_1.csv",
    index=False
)

casos_regra_2.to_csv(
    pasta_outputs / "nivel_1_casos_regra_2.csv",
    index=False
)

clientes_sinalizados.to_csv(
    pasta_outputs / "nivel_1_clientes_sinalizados.csv",
    index=False
)

comparacao_prompts.to_csv(
    pasta_outputs / "nivel_1_comparacao_prompts.csv",
    index=False
)

print("Arquivos CSV exportados.")

Arquivos CSV exportados.


In [176]:
resultado_pareceres = {
    "cliente_escolhido": cliente_escolhido,
    "modelo": modelo,
    "versao_1": {
        "prompt": prompt_versao_1,
        "validacao": resultado_validacao_v1,
        "metricas": metricas_versao_1
    },
    "versao_2": {
        "prompt": prompt_versao_2,
        "validacao": resultado_validacao_v2,
        "metricas": metricas_versao_2
    }
}

caminho_pareceres = (
    pasta_outputs / "nivel_1_pareceres_llm.json"
)

with open(
    caminho_pareceres,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultado_pareceres,
        arquivo,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print("Pareceres exportados para:", caminho_pareceres)

Pareceres exportados para: c:\Users\maria\Desktop\desafio-estagio-ia\outputs\nivel_1_pareceres_llm.json


In [177]:
print("VERSÃO 1")
print(
    json.dumps(
        parecer_versao_1,
        ensure_ascii=False,
        indent=2
    )
)

print("\nVERSÃO 2")
print(
    json.dumps(
        parecer_versao_2,
        ensure_ascii=False,
        indent=2
    )
)

print("\nCOMPARAÇÃO DOS PROMPTS")
display(comparacao_prompts)

print("\nPasta de resultados:", pasta_outputs)

VERSÃO 1
{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Fracionamento de valores em um mesmo dia para atingir montante superior a BRL 50.000,00 sem que nenhuma operação individual atinja BRL 20.000,00",
  "red_flags": [
    "Realização de 3 operações na mesma data (2026-03-09)",
    "Soma das operações na data atingiu BRL 54.200,00, superando o limite de BRL 50.000,00",
    "Nenhuma das operações individuais na data atingiu ou ultrapassou o patamar de BRL 20.000,00 (maior operação de BRL 18.800,00)"
  ],
  "justificativa": "Trata-se de um parecer preliminar, visto que não há informações suficientes para uma conclusão definitiva. O cliente CLI-A-1 foi sinalizado pela Regra 1 em decorrência de um padrão operacional na data 2026-03-09, onde foram registradas 3 transferências enviadas com soma total de BRL 54.200,00, sendo que individualmente nenhuma operação atingiu BRL 20.000,00. Há necessidade de análise adicional para verificar a motivação comercial das transações."
}

VERSÃO 2
{

,versao,modelo,json_valido,nivel_risco,quantidade_red_flags,tempo_segundos,tentativas,tokens_entrada,tokens_saida,tokens_total
0,Versão 1,gemini-3.5-flash-lite,True,médio,3,1.94,1,1057,331,1388
1,Versão 2,gemini-3.5-flash-lite,True,médio,3,18.65,1,1092,324,1416



Pasta de resultados: c:\Users\maria\Desktop\desafio-estagio-ia\outputs
